# Annotating descriptors using syntactic analysis (spaCy)


## 1. Libraries import

In [1]:
import re
import pandas as pd
import spacy
import json
import unicodedata
from pathlib import Path
from collections import defaultdict
from collections import Counter
import ast

c:\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.3.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


## 2. Data and model import

In [2]:
FICHIER_ENTREE = "voies_par_longueur_annotation_cotationfr.csv"

# File created for the controlled test.
FICHIER_SORTIE_TEST = "annotation_spacy_test_50_occurrences.csv"

# Final output file, used when SAMPLE_SIZE = None.
FICHIER_SORTIE_COMPLET = "annotation_spacy_occurrences_detaillees.csv"

In [3]:
def charger_modele_spacy(nom_modele):
    """
    Load the spaCy model and check that syntactic parsing is available.
    """
    try:
        nlp = spacy.load(nom_modele)
    except OSError as exc:
        raise OSError(
            f"spaCy model '{nom_modele}' is not installed.\n"
            "In Google Colab, run this command in a cell:\n"
            f"    !python -m spacy download {nom_modele}\n"
            "Then rerun the script."
        ) from exc

    if not nlp.has_pipe("parser"):
        raise RuntimeError(
            f"Model '{nom_modele}' does not include a parser component. "
            "A model with syntactic parsing is required to suggest "
            "associations between terms and descriptors."
        )

    return nlp


def preparer_echantillon(df, taille, graine):
    """
    Select a reproducible sample of rows from the CSV.

    Sampling is performed on the source descriptions, before detecting
    terms. This means descriptions with no terms are also represented in
    the coverage check, even if they do not produce an output row.
    """
    if taille is None:
        return df.copy()

    if taille <= 0:
        raise ValueError("SAMPLE_SIZE must be a positive integer or None.")

    n = min(taille, len(df))
    return df.sample(n=n, random_state=graine).sort_index().copy()


## 3. Defining parameters

In [4]:
# 50 = process 50 descriptions selected randomly and reproducibly.
# None = process all descriptions.
SAMPLE_SIZE = None
RANDOM_SEED = 42

# Number of spaCy tokens before and after the detected term.
FENETRE_TOKENS = 3

# French model with syntactic parsing.
MODELE_SPACY = "fr_core_news_md"

## 4. Dictionary of forms to identify

In [5]:
# The keys are canonical terms. The lists contain the forms to detect in the text. The dictionary is intentionally kept as provided; 
# the script does not add terms on its own.

FORMES = {
    "dévers": [
        "dévers", "déversante", "déversant", "déversantes", "déversants",
        "déversé", "déversée", "déversées", "déversés", "toit", "toits"
    ],
    "surplomb": [
        "surplombant", "surplombante", "surplombé", "surplomb", "surplombs",
        "bombé", "bombés", "bombée", "bombées"
    ],
    "dalle": ["dalle", "dalles", "adhérence", "adhérences"],
    "faille": ["faille", "failles"],
    "épaule": ["épaule", "épaules", "épaulement", "épaulements"],
    "dièdre": ["dièdre", "dièdres"],
    "cheminée": ["cheminée", "cheminées"],
    "vire": ["vire", "vires", "virette", "virettes"],
    "terrasse": ["terrasse", "terrasses", "plateforme", "plateformes"],
    "fissure": ["fissure", "fissures"],
    "traversée": [
        "traversée", "traversées", "traverser", "traverse", "traversant"
    ],
    "réglette": ["réglette", "réglettes"],
    "bac": [
        "bac", "grosse prise", "grosses prises", "bonne prise", "bonnes prises"
    ],
    "plat": ["plat", "plats", "replat", "replats"],
    "pince": ["pince", "pinces"],
    "mono_doigt": ["mono-doigt", "monodoigt", "mono-doigts", "monodoigts"],
    "bi_doigt": ["bi-doigt", "bidoigt", "bi-doigts", "bidoigts"],
    "inversee": [
        "inversée", "inversées", "inversé", "inversés", "inverse", "inverses"
    ],
    "crochet": ["crochet", "crochets"],
    "trou": ["trou", "trous"],
    "goutte_eau": ["goutte d'eau", "gouttes d'eau"],
    "petite_prise": ["petite prise", "petites prises"],
    "écailles": ["écaille", "écailles"],
    "arête": ["arête", "arêtes"],
    "pilier": ["pilier", "piliers"],
    "couloir": ["couloir", "couloirs"],
    "cannelure": [
        "cannelure", "cannelures", "cannelé", "cannelée", "cannelés",
        "cannelées", "canelée", "canelé"
    ],
    "éperon": ["éperon", "éperons"],
    "contrefort": ["contrefort", "contreforts"],
    "corniche": ["corniche", "corniches"],
    "fissure-dièdre": [
        "fissure-dièdre", "fissures-dièdre", "fissure-dièdres",
        "fissures-dièdres"
    ],
}

## 5. Defining helper functions

In [6]:
def normaliser_texte(texte):
    """
    Normalize text to compare forms regardless of case.

    Accents are preserved, so two different
    spellings are not arbitrarily treated as identical. Curly apostrophes are standardized.
    """
    texte = str(texte).replace("’", "'").replace("‘", "'")
    return texte.casefold()


def compiler_motif_formes(dictionnaire):
    """
    Build a regex pattern that detects all forms in the dictionary.

    Longer forms come first so that, when they start
    at the same position, an expression such as « fissure-dièdre » is recognized
    before a shorter form.

    Unicode boundaries prevent detecting « trou » in
    « trousses » or « bac » in the middle of another word.
    """
    formes_aplaties = []

    for terme_canonique, liste_formes in dictionnaire.items():
        for forme in liste_formes:
            formes_aplaties.append((terme_canonique, forme))

    # Longest forms first, then stable alphabetical order.
    formes_aplaties.sort(key=lambda item: (-len(item[1]), item[1].casefold()))

    alternatives = []
    for _, forme in formes_aplaties:
        # Curly apostrophes are treated as apostrophes.
        forme_regex = re.escape(forme).replace("’", "['’]").replace("'", "['’]")
        alternatives.append(forme_regex)

    motif = re.compile(
        r"(?<!\w)(?P<forme>" + "|".join(alternatives) + r")(?!\w)",
        flags=re.IGNORECASE | re.UNICODE
    )
    return motif


def construire_index_formes(dictionnaire):
    """
    Create a form -> canonical terms lookup table.

    A form could theoretically belong to several keys; in that case, keep
    all keys and report this ambiguity in the result.
    """
    index = {}

    for terme_canonique, liste_formes in dictionnaire.items():
        for forme in liste_formes:
            cle = normaliser_texte(forme)
            index.setdefault(cle, [])
            if terme_canonique not in index[cle]:
                index[cle].append(terme_canonique)

    return index


def trouver_colonne_id(df):
    """Return the name of the first recognized identifier column, if present."""
    candidats = ["document_id", "id", "ID", "route_id"]
    for colonne in candidats:
        if colonne in df.columns:
            return colonne
    return None


def extraire_occurrences(texte, motif, index_formes):
    """
    Find lexical occurrences in the original text.

    Return one occurrence for each regex match. The detected forms
    remain distinct; any overlaps are reported later.
    """
    occurrences = []

    if not isinstance(texte, str) or not texte.strip():
        return occurrences

    for match in motif.finditer(texte):
        forme = match.group("forme")
        cle_forme = normaliser_texte(forme)
        termes = index_formes.get(cle_forme, [])

        if not termes:
            # Safety case: the form should always be in the index.
            continue

        occurrences.append({
            "forme_detectee": forme,
            "termes_canoniques": termes,
            "position_debut": match.start(),
            "position_fin": match.end(),
        })

    return occurrences


def indices_tokens_pour_occurrence(doc, debut_caractere, fin_caractere):
    """
    Align an occurrence found in the raw text with spaCy tokens.

    Character offsets ensure the correct occurrence is retained even
    when the form appears several times in the description.
    """
    tokens = [
        token for token in doc
        if token.idx < fin_caractere
        and token.idx + len(token.text) > debut_caractere
    ]

    if not tokens:
        return None, None

    return tokens[0].i, tokens[-1].i


def obtenir_fenetre(doc, debut_token, fin_token, taille=5):
    """
    Return the context consisting of N tokens before and N tokens after.

    The detected term itself is not included in the five tokens on either
    side; the full expression is displayed in the center of the window.
    """
    debut_fenetre = max(0, debut_token - taille)
    fin_fenetre = min(len(doc), fin_token + taille + 1)

    tokens = list(doc[debut_fenetre:fin_fenetre])
    return " ".join(token.text for token in tokens)


def phrase_autour_occurrence(doc, debut_token, fin_token):
    """
    Return the spaCy sentence containing the occurrence, when available.
    """
    token = doc[debut_token]
    try:
        return token.sent.text.strip()
    except Exception:
        return " ".join(t.text for t in doc[max(0, debut_token - 8):fin_token + 9])



## 6. Extraction

In [7]:
def token_est_qualificatif(token):
    """
    Indicate whether a token can be a descriptive qualifier.
    """
    lemmes_exclus = {
        "typer", "nouveau", "partir", "continuer", "traverser",
        "suivre", "emprunter", "pousser", "caractériser",
        "suivant", "supérieur", "gauche", "droite", "long",
        "rocheux", "bord",
    }

    if token.lemma_.casefold() in lemmes_exclus:
        return False

    if token.pos_ == "ADJ":
        return True

    # spaCy sometimes tags rare adjectives as NOUN
    # (« végétative », « Jolies »). If the syntactic relation indicates
    # an « adjectival modifier », trust the relation.
    if token.pos_ == "NOUN" and token.dep_ == "amod":
        return True

    # Depending on the French model's analysis, some participles (« couché »,
    # « soutenue ») may be tagged VERB rather than ADJ.
    if token.pos_ in {"VERB", "AUX"}:
        forme_verbale = token.morph.get("VerbForm")
        if "Part" in forme_verbale and token.dep_ in {
            "amod", "acl", "acl:relcl", "ROOT", "conj"
        }:
            return True

    return False


NOMS_QUALIFICATIFS = {
    "trou", "trous", "adhérence", "adhérences", "dülfer", "dulfer",
    "cannelure", "cannelures", "bac", "bacs", "écaille", "écailles",
    "fissure", "fissures", "réglette", "réglettes", "végétation",
}

def qualificatif_nominal(token, indices_terme):
    """
    Nouns on an allowlist that can qualify the term:
    « dalle à trous », « fissure dülfer », « dalle en adhérence ».
    Other nouns (gauche, piton, mètres, bord...) are never
    qualifiers: they produced most of the noise.
    """
    if token.lemma_.casefold() not in NOMS_QUALIFICATIFS:
        return False
    if token.head.i in indices_terme:
        return True
    if token.head.pos_ == "ADP" and token.head.head.i in indices_terme:
        return True
    return False


def trouver_qualificatifs(doc, debut_token, fin_token, taille=3):
    """
    Find candidates, prioritizing direct syntactic relations.

    Improvements:
    - accepts some adjectival participles, such as « couché » and « soutenue »;
    - groups intensifiers/comparatives (« peu raide », « plus soutenue »);
    - adds a very local proximity fallback for adjectives immediately
      before/after the term, even if the parser analyzed the relation incorrectly;
    - treats « qui est plus propre » as a possible description of the noun
      immediately preceding it;
    - does not use proximity alone to associate an adjective with a
      verbal comme « traverser » : cela évite notamment « gros » -> « traversée »
      in « deux gros becquets et traverser ».
    """
    debut_fenetre = max(0, debut_token - taille)
    fin_fenetre = min(len(doc), fin_token + taille + 1)

    # Targeted extension for a relative clause such as « l'arête qui est plus propre ».
    fin_etendue = min(len(doc), fin_token + 7)
    tokens_fenetre = list(doc[debut_fenetre:max(fin_fenetre, fin_etendue)])
    tokens_terme = list(doc[debut_token:fin_token + 1])
    indices_terme = {t.i for t in tokens_terme}
    cible_est_verbe_traverser = any(
        t.pos_ in {"VERB", "AUX"} and t.lemma_.casefold() == "traverser"
        for t in tokens_terme
    )

    candidats_forts = []
    candidats_fenetre = []
    vus = set()

    def ajouter(item, direct):
        cle = (item["position_debut"], item["position_fin"], item["texte"])
        if cle in vus:
            return
        vus.add(cle)
        item["lien_syntaxique_direct"] = bool(direct)
        item.setdefault("admissible_attribution", bool(direct))
        (candidats_forts if direct else candidats_fenetre).append(item)


    for token in tokens_fenetre:
        if token.i in indices_terme or token.is_punct or token.like_num:
            continue
        if not (
            token_est_qualificatif(token)
            or qualificatif_nominal(token, indices_terme)
        ):
            continue

        # Avoid distant descriptors included only by the special
        # « qui est plus ... » extension, unless that structure is present.
        distance = min(
            abs(token.i - debut_token),
            abs(token.i - fin_token)
        )
        relative_plus_propre = (
            token.lemma_.casefold() == "propre"
            and token.i > fin_token
            and any(t.lower_ == "qui" for t in doc[fin_token + 1:token.i])
            and any(t.lower_ == "est" for t in doc[fin_token + 1:token.i])
            and any(t.lower_ == "plus" for t in doc[fin_token + 1:token.i])
        )
        if distance > taille and not relative_plus_propre:
            continue
        
        direct_syntaxe = (
            token.head.i in indices_terme
            or any(t.head.i == token.i for t in tokens_terme)
            or (
                not cible_est_verbe_traverser
                and any(
                    t.dep_ == "conj" and token.head.i == t.head.i
                    for t in tokens_terme
                )
            )
        )

        # Local fallback: an adjective immediately before/after the term, e.g.
        # « franche traversée », « jolies adhérences », « vire confortable »,
        # « dièdre couché ». Do not apply it to a verbal target such as « traverser ».
        adjacent_local = (
            distance <= taille
            and not cible_est_verbe_traverser
            and token.pos_ in {"ADJ", "VERB"}
        )

        # An adjective that modifies another noun (e.g. « gros becquets ») must not
        # be picked up as a descriptor of a nearby term based on proximity.
        tete_autre_nom = (
            token.head.pos_ in {"NOUN", "PROPN"}
            and token.head.i not in indices_terme
            and token.head.i != token.i
        )
        if tete_autre_nom and not direct_syntaxe:
            adjacent_local = False

        # Actual syntactic links remain in the syntax column. Associations
        # based solely on proximity remain window candidates;
        # the explicit « qui est plus propre » rule is retained
        # as a strong association because it identifies a descriptive relative clause.
        direct = direct_syntaxe or relative_plus_propre
        candidat_local = adjacent_local and not direct

        # Group adverbs that modify the adjective:
        # « peu raide », « plus soutenue », « très facile ».
        modificateurs = [
            child for child in token.children
            if child.pos_ == "ADV" and child.dep_ in {"advmod", "fixed"}
        ]

        # Targeted lexical fallback: in some models, « plus » is not
        # attached as advmod to « soutenue/propre/raide ».
        if not modificateurs and token.i > 0 and doc[token.i - 1].lower_ in {
            "plus", "moins", "très", "peu", "assez", "trop"
        }:
            modificateurs = [doc[token.i - 1]]

        modificateurs.sort(key=lambda t: t.i)
        debut = min([token.idx] + [t.idx for t in modificateurs])
        fin = max(
            [token.idx + len(token.text)]
            + [t.idx + len(t.text) for t in modificateurs]
        )
        expression = doc.char_span(debut, fin, alignment_mode="expand")
        texte_expr = expression.text if expression is not None else token.text
        lemme_expr = " ".join(
            [t.lemma_ for t in modificateurs] + [token.lemma_]
        ).strip()

        ajouter({
            "texte": texte_expr,
            "lemme": lemme_expr,
            "relation": (
                f"groupe_adjectival:{token.dep_}"
                if direct else "proximite_locale"
            ),
            "position_debut": debut,
            "position_fin": fin,
            "exception_coordination": any(
                t.dep_ == "conj" and t.head.i == token.head.i
                for t in tokens_terme
            ) or any(child.dep_ == "conj" for child in token.head.children),
            # Window candidates remain visible for auditing, but can
            # be selected only if they are very close or explicitly
            # recognized by a structural rule.
            "admissible_attribution": bool(
                direct_syntaxe or adjacent_local or relative_plus_propre
            ),
        }, direct)
    # Common descriptive expressions that are not adjectives:
    # « traversée en ascendance », y compris « en ascendance à droite ».
    if any("travers" in t.lemma_.casefold() or "travers" in t.text.casefold()
           for t in tokens_terme):
        limite = min(fin_token + 6, len(doc) - 1)
        for i in range(debut_token, limite):
            if (
                doc[i].lower_ == "en"
                and i + 1 < len(doc)
                and "ascendance" in (
                    doc[i + 1].lemma_.casefold() + " " + doc[i + 1].text.casefold()
                )
            ):
                ajouter({
                    "texte": f"{doc[i].text} {doc[i + 1].text}",
                    "lemme": "en ascendance",
                    "relation": "expression_descriptive_connue",
                    "position_debut": doc[i].idx,
                    "position_fin": doc[i + 1].idx + len(doc[i + 1].text),
                    "exception_coordination": False,
                    "admissible_attribution": True,
                }, True)
                break

    return candidats_forts, candidats_fenetre


def resoudre_exclusivite_qualificatifs(resultats):
    """Assign each candidate qualifier to a single term by default.

    All candidates remain visible in the audit columns. The
     `qualificatifs_retenus` column contains only the final assignment.
    If the same candidate segment appears for multiple terms, priority goes
    to the direct syntactic link, then to the shortest token distance.
    """
    import json

    candidats_par_cle = {}
    candidats_par_ligne = []
    for idx, resultat in enumerate(resultats):
        forts = json.loads(resultat.get("qualificatifs_syntaxe", "[]") or "[]")
        fenetre = json.loads(resultat.get("qualificatifs_fenetre", "[]") or "[]")
        candidats = []
        for item in forts:
            candidats.append((item, True))
        for item in fenetre:
            candidats.append((item, False))
        candidats_par_ligne.append(candidats)
        cible = (resultat["position_debut"] + resultat["position_fin"]) / 2
        for item, direct in candidats:
            # A weak candidate may remain visible in the window column
            # without being automatically assigned to a term.
            if not item.get("admissible_attribution", True):
                continue
            cle = (item.get("position_debut"), item.get("position_fin"), item.get("texte"))
            distance = abs((item.get("position_debut", 0) + item.get("position_fin", 0)) / 2 - cible)
            score = (0 if direct or item.get("lien_syntaxique_direct") else 1, distance)
            candidats_par_cle.setdefault(cle, []).append((score, idx, item, direct))

    retenus = [[] for _ in resultats]
    conflits = [False for _ in resultats]
    for cle, propositions in candidats_par_cle.items():
        # Coordination exceptions are not generalized: if spaCy
        # does not provide a clear direct link, keep a single owner.
        propositions.sort(key=lambda x: (x[0], x[1]))
        # Sharing is allowed only if the analysis explicitly indicates
        # coordination and the competing candidates are syntactic.
        partage_coordonne = (
            len(propositions) > 1
            and all(p[0][0] == 0 for p in propositions)
            and any(p[2].get("exception_coordination", False) for p in propositions)
        )
        if partage_coordonne:
            for proposition in propositions:
                retenus[proposition[1]].append(proposition[2])
        else:
            meilleur = propositions[0]
            retenus[meilleur[1]].append(meilleur[2])
        if len(propositions) > 1 and not partage_coordonne:
            for proposition in propositions:
                conflits[proposition[1]] = True

    for idx, resultat in enumerate(resultats):
        resultat["qualificatifs_retenus"] = json.dumps(
            sorted(retenus[idx], key=lambda x: x.get("position_debut", 0)),
            ensure_ascii=False
        )
        if conflits[idx]:
            resultat["a_verifier"] = True
            motif = resultat.get("motif_verification", "") or ""
            ajout = "un même qualificatif candidat concernait plusieurs termes ; attribution unique effectuée"
            resultat["motif_verification"] = (motif + " ; " if motif else "") + ajout

    return resultats

def detecter_ambiguite(
    occurrence,
    toutes_occurrences,
    qualificatifs_forts,
    qualificatifs_fenetre
):
    """
    Produce transparent audit flags.

    The rules are intentionally conservative:
    - form belonging to multiple canonical terms;
    - adjectival form potentially describing another feature;
    - occurrence overlapping another occurrence;
    - qualifiers found only in the window, without a strong syntactic link;
    - no candidate qualifier found.
    """
    motifs = []

    if len(occurrence["termes_canoniques"]) > 1:
        motifs.append("forme associée à plusieurs termes canoniques")

    debut = occurrence["position_debut"]
    fin = occurrence["position_fin"]

    for autre in toutes_occurrences:
        if autre is occurrence:
            continue
        chevauchement = (
            debut < autre["position_fin"]
            and autre["position_debut"] < fin
        )
        if chevauchement:
            motifs.append(
                "chevauchement avec une autre forme du dictionnaire "
                f"({autre['forme_detectee']})"
            )
            break

    forme_normalisee = normaliser_texte(occurrence["forme_detectee"])
    formes_adjectivales_ambiguës = {
        "cannelé", "cannelée", "cannelés", "cannelées", "canelé", "canelée",
        "surplombant", "surplombante", "déversant", "déversante",
        "déversé", "déversée", "inversé", "inversée"
    }
    if forme_normalisee in formes_adjectivales_ambiguës:
        motifs.append(
            "forme pouvant être un qualificatif d'un autre élément selon le contexte"
        )

    if qualificatifs_fenetre and not qualificatifs_forts:
        motifs.append(
            "qualificatif(s) présent(s) dans la fenêtre sans lien syntaxique fort"
        )

    if not qualificatifs_forts and not qualificatifs_fenetre:
        motifs.append("aucun qualificatif candidat détecté dans la fenêtre")

    return bool(motifs), " ; ".join(dict.fromkeys(motifs))


## 7. Annotating a description

In [8]:
def annoter_description(
    texte,
    doc,
    motif_formes,
    index_formes,
    fenetre_tokens=5
):
    """
    Return a list of detailed rows: one row per detected occurrence.

    Candidates are kept separate from the selected associations.
    A unique assignment is then resolved at the description level.
    """
    occurrences = extraire_occurrences(texte, motif_formes, index_formes)
    resultats = []

    for occurrence in occurrences:
        debut_token, fin_token = indices_tokens_pour_occurrence(
            doc,
            occurrence["position_debut"],
            occurrence["position_fin"]
        )

        if debut_token is None:
            # The occurrence could not be aligned with spaCy tokens.
            contexte = texte[
                max(0, occurrence["position_debut"] - 40):
                min(len(texte), occurrence["position_fin"] + 40)
            ]
            qualificatifs_forts = []
            qualificatifs_fenetre = []
            phrase = contexte
            erreur_alignement = True
        else:
            contexte = obtenir_fenetre(
                doc, debut_token, fin_token, taille=fenetre_tokens
            )
            phrase = phrase_autour_occurrence(doc, debut_token, fin_token)
            qualificatifs_forts, qualificatifs_fenetre = trouver_qualificatifs(
                doc, debut_token, fin_token, taille=fenetre_tokens
            )
            erreur_alignement = False

        a_verifier, motif_verification = detecter_ambiguite(
            occurrence,
            occurrences,
            qualificatifs_forts,
            qualificatifs_fenetre
        )

        if erreur_alignement:
            a_verifier = True
            motif_verification = (
                motif_verification + " ; " if motif_verification else ""
            ) + "alignement occurrence / tokens spaCy impossible"

        # One row per canonical term, while retaining the detected form.
        for terme_canonique in occurrence["termes_canoniques"]:
            resultats.append({
                "terme_canonique": terme_canonique,
                "forme_detectee": occurrence["forme_detectee"],
                "position_debut": occurrence["position_debut"],
                "position_fin": occurrence["position_fin"],
                "fenetre_contexte": contexte,
                "phrase_contexte": phrase,
                "qualificatifs_syntaxe": json.dumps(
                    qualificatifs_forts, ensure_ascii=False
                ),
                "qualificatifs_fenetre": json.dumps(
                    qualificatifs_fenetre, ensure_ascii=False
                ),
                "a_verifier": a_verifier,
                "motif_verification": motif_verification,
            })

    return resultats



## 8. Annotating all descriptions

In [9]:
def main():
    print("=" * 72)
    print("spaCy ANNOTATION OF TERMS AND DESCRIPTORS")
    print("=" * 72)

    chemin_entree = Path(FICHIER_ENTREE)
    if not chemin_entree.exists():
        raise FileNotFoundError(
            f"File not found: {chemin_entree.resolve()}\n"
            "Place the CSV in the current folder or update FICHIER_ENTREE."
        )

    print(f"\nReading file: {chemin_entree.resolve()}")
    df = pd.read_csv(chemin_entree, low_memory=False)

    if "description" not in df.columns:
        raise KeyError(
            "The 'description' column was not found in the CSV. "
            f"Available columns: {list(df.columns)}"
        )

    print(f"Total number of source rows: {len(df):,}")
    print(f"Number of source columns: {len(df.columns)}")

    colonne_id = trouver_colonne_id(df)
    if colonne_id:
        print(f"Identifier column used: {colonne_id}")
    else:
        print("No recognized ID column; the source index will be retained.")

    # Add stable provenance: the CSV row number, starting at 1.
    # This number is created before sampling and therefore remains linked to the
    # original file even when only a subset is processed.
    df = df.copy()
    df["source_ligne"] = range(1, len(df) + 1)

    df_test = df.head(SAMPLE_SIZE).copy()

    if SAMPLE_SIZE is None:
        print("Mode: processing all descriptions.")
        fichier_sortie = FICHIER_SORTIE_COMPLET
    else:
        print(
            f"Test mode: {len(df_test)} descriptions sampled "
            f"(random seed = {RANDOM_SEED})."
        )
        fichier_sortie = FICHIER_SORTIE_TEST

    print(f"\nLoading spaCy model: {MODELE_SPACY}")
    nlp = charger_modele_spacy(MODELE_SPACY)

    motif_formes = compiler_motif_formes(FORMES)
    index_formes = construire_index_formes(FORMES)

    lignes_detaillees = []
    nb_descriptions_avec_match = 0
    nb_occurrences = 0
    nb_descriptions_sans_match = 0

    # nlp.pipe is more efficient than calling nlp() separately for each
    # description, especially across tens of thousands of rows.
    descriptions = [
        "" if pd.isna(valeur) else str(valeur)
        for valeur in df_test["description"].tolist()
    ]

    print("\nAnalysis in progress...")
    for position, (index_df, doc) in enumerate(
        zip(df_test.index.tolist(), nlp.pipe(descriptions, batch_size=64)),
        start=1
    ):
        ligne = df_test.loc[index_df]
        texte = descriptions[position - 1]

        resultats = annoter_description(
            texte=texte,
            doc=doc,
            motif_formes=motif_formes,
            index_formes=index_formes,
            fenetre_tokens=FENETRE_TOKENS
        )

        resultats = resoudre_exclusivite_qualificatifs(resultats)

        if resultats:
            nb_descriptions_avec_match += 1
            nb_occurrences += len(resultats)
        else:
            nb_descriptions_sans_match += 1

        for resultat in resultats:
            ligne_sortie = {
                "source_ligne": int(ligne["source_ligne"]),
                "index_dataframe": int(index_df),
                "document_id": (
                    ligne[colonne_id]
                    if colonne_id is not None else None
                ),
                "texte_original": texte,
            }
            ligne_sortie.update(resultat)
            lignes_detaillees.append(ligne_sortie)

        if position % 10 == 0 or position == len(df_test):
            print(
                f"  Progress: {position}/{len(df_test)} descriptions "
                f"| occurrences retained: {len(lignes_detaillees)}"
            )

    colonnes_sortie = [
        "source_ligne",
        "index_dataframe",
        "document_id",
        "texte_original",
        "terme_canonique",
        "forme_detectee",
        "position_debut",
        "position_fin",
        "fenetre_contexte",
        "phrase_contexte",
        "qualificatifs_syntaxe",
        "qualificatifs_fenetre",
        "qualificatifs_retenus",
        "a_verifier",
        "motif_verification",
    ]

    resultat_df = pd.DataFrame(lignes_detaillees, columns=colonnes_sortie)
    resultat_df.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 72)
    print("SUMMARY")
    print("=" * 72)
    print(f"Descriptions processed: {len(df_test):,}")
    print(f"Descriptions with at least one term: {nb_descriptions_avec_match:,}")
    print(f"Descriptions with no detected term: {nb_descriptions_sans_match:,}")
    print(f"Detailed rows produced: {len(resultat_df):,}")

    if not resultat_df.empty:
        print(
            "Rows flagged for review: "
            f"{int(resultat_df['a_verifier'].fillna(False).sum()):,}"
        )
        print("\nOccurrences by canonical term:")
        print(
            resultat_df["terme_canonique"]
            .value_counts()
            .to_string()
        )
    else:
        print(
            "\nNo occurrences were found in the processed descriptions. "
            "Check that the CSV and the 'description' column are correct."
        )

    print(f"\nDetailed file created: {Path(fichier_sortie).resolve()}")

    if SAMPLE_SIZE is not None:
        print(
            "\nNext step: open and review the test CSV. "
            "If the results are satisfactory, replace SAMPLE_SIZE = 50 "
            "with SAMPLE_SIZE = None, then rerun the script to process "
            "all descriptions. The full run will write to "
            f"'{FICHIER_SORTIE_COMPLET}'."
        )


if __name__ == "__main__":
    main()


spaCy ANNOTATION OF TERMS AND DESCRIPTORS

Reading file: C:\Users\lunag\Documents\Etudes\Etudes Sup\CPES_L3\ProjetRecherche\Code\Version20\CodePropre\2.TraitementDonnees\English\voies_par_longueur_annotation_cotationfr.csv
Total number of source rows: 42,861
Number of source columns: 36
Identifier column used: document_id
Mode: processing all descriptions.

Loading spaCy model: fr_core_news_md

Analysis in progress...
  Progress: 10/42861 descriptions | occurrences retained: 19
  Progress: 20/42861 descriptions | occurrences retained: 33
  Progress: 30/42861 descriptions | occurrences retained: 44
  Progress: 40/42861 descriptions | occurrences retained: 63
  Progress: 50/42861 descriptions | occurrences retained: 72
  Progress: 60/42861 descriptions | occurrences retained: 88
  Progress: 70/42861 descriptions | occurrences retained: 109
  Progress: 80/42861 descriptions | occurrences retained: 124
  Progress: 90/42861 descriptions | occurrences retained: 143
  Progress: 100/42861 desc